In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import os
import cv2
import json
import glob
import re
import shutil

llista_lleida = [
    "01_James_Batemon", "02_Corey_Walden", "03_Melvin_Ejim",
    "04_Caleb_Agada", "05_Oriol_Pauli", "06_Dani_Garcia",
    "07_Gyorgy_Goloman", "12_Atoumane_Diagne", "14_Mikel_Sanz",
     "22_John_Shurna", "25_Cameron_Krutwig"
]
llista_joventut = [
    "01_Y_Kraag",
    "09_R_Rubio",
    "11_J_Parker",
    "12_L_Hakanson",
    "23_M_Ruzic",
    "24_C_Hunt",
    "35_S_Birgander",
    "88_A_Hanga"
]

In [6]:
!pip install yt-dlp
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4" "https://www.3cat.cat/3cat/lleida-joventut-lliga-j23/video/6394037/" -o "partit_lleida_V2.mp4"

RUTA_VIDEO = 'partit_lleida_V2.mp4'

def extreure_frame_fons(ruta_video, segon):
    """
    Obre el vídeo dinàmicament, calcula el frame corresponent al segon indicat
    i el retorna convertit a format RGB per a Matplotlib.
    """
    if not os.path.exists(ruta_video):
        print(f"No s'ha trobat el vídeo a la ruta '{ruta_video}'.")
        videos = glob.glob("**/*.mp4", recursive=True) + glob.glob("**/*.avi", recursive=True)
        if videos:
            ruta_video = videos[0]
            print(f"S'utilitzarà el vídeo : '{ruta_video}'")
        else:
            print(" No s'ha trobat cap vídeo de fons.")
            return None

    cap = cv2.VideoCapture(ruta_video)
    if not cap.isOpened():
        print(" No s'ha pogut instanciar el pont de lectura del vídeo.")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25.0

    num_frame = int(segon * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, num_frame)

    ret, frame = cap.read()
    cap.release()

    if ret:
        print(f" S'ha extret correctament el frame del segon {segon} (Fotograma {num_frame}).")
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    else:
        print(f" No s'ha pogut llegir el frame {num_frame}. S'usarà fons fosc neutre.")
        return None

FRAME_FONS = extreure_frame_fons(RUTA_VIDEO, segon=90)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.9 MB/s eta 0:00:00
[CCMA] Extracting URL: https://www.3cat.cat/3cat/lleida-joventut-lliga-j23/video/6394037/
[CCMA] 6394037: Downloading JSON metadata
[CCMA] 6394037: Downloading MPD manifest
[info] 6394037: Downloading 1 format(s): dash-10feb6f6-ba4c-483f-8caa-b6ec7eafb660+dash-a416bd18-b3f1-42b5-9f56-603901a4b812
[dashsegments] Total fragments: 1786
[download] Destination: partit_lleida_V2.fdash-10feb6f6-ba4c-483f-8caa-b6ec7eafb660.mp4
[download] 100% of    3.24GiB in 00:12:08 at 4.56MiB/s
[dashsegments] Total fragments: 1786
[download] Destination: partit_lleida_V2.fdash-a416bd18-b3f1-42b5-9f56-603901a4b812.m4a
[download] 100% of  108.26MiB in 00:06:46 at 272.44KiB/s
[Merger] Merging formats into "partit_lleida_V2.mp4"
Deleting original file partit_lleida_V2.fdash-a416bd18-b3f1-42b5-9f56-603901a4b812.m4a (pass -k to keep)
Deleting original file 

In [10]:
import os
import glob
import re
import json

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    patrons = [
        "/kaggle/input/**/posicions_jugadors*.json",
        "/kaggle/input/**/*.json"
    ]
elif 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    patrons = [
        "/content/**/posicions_jugadors*.json",
        "/content/**/*.json",
        "posicions_jugadors*.json",
        "**/*.json"
    ]
else:
    patrons = [
        "posicions_jugadors*.json",
        "**/posicions_jugadors*.json",
        "*.json",
        "**/*.json"
    ]

fitxers_json = []
for patro in patrons:
    fitxers_json = glob.glob(patro, recursive=True)
    if fitxers_json:
        break

def order(ruta_arxiu):
    nums = re.findall(r'\d+', os.path.basename(ruta_arxiu))
    return int(nums[0]) if nums else 0

fitxers_json = sorted(list(set(fitxers_json)), key=order)

dades_posicions = {}

print("=" * 70)
print(f"S'han trobat {len(fitxers_json)} fitxers JSON a l'entorn. Iniciant la fusió...")
print("-" * 70)

for fitxer in fitxers_json:
    nom_breu = os.path.basename(fitxer)

    if "checkpoint" in nom_breu.lower():
        continue

    with open(fitxer, 'r', encoding='utf-8') as f:
        try:
            contingut = json.load(f)
            if isinstance(contingut, dict):
                nous_punts_comptador = 0
                for jugador_nom, coordenades in contingut.items():
                    if jugador_nom not in dades_posicions:
                        dades_posicions[jugador_nom] = []

                    if isinstance(coordenades, list):
                        dades_posicions[jugador_nom].extend(coordenades)
                        nous_punts_comptador += len(coordenades)
                print(f" -> Fusionat amb èxit: '{nom_breu}' (+{nous_punts_comptador} coordenades).")
        except Exception as e:
            print(f" ->  Error de lectura a '{nom_breu}': {e}")

print("=" * 70)
print(f" Procés de fusió finalitzat.")
print(f"Tenim un total de {len(dades_posicions.keys())} identitats úniques unificades.")
print("Resum de coordenades acumulades per jugador (en viu):")
print("-" * 70)
for jugador, coords in sorted(dades_posicions.items()):
    print(f" - {jugador:25s} -> {len(coords):6d} coordenades totals recollides.")
print("=" * 70)

S'han trobat 12 fitxers JSON a l'entorn. Iniciant la fusió...
----------------------------------------------------------------------
 -> Fusionat amb èxit: 'posicions_jugadors2.json' (+79088 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors3.json' (+97617 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors4.json' (+119577 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors5.json' (+136977 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors6.json' (+164146 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors7.json' (+182413 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors8.json' (+210937 coordenades).
 -> Fusionat amb èxit: 'posicions_jugadors9.json' (+230282 coordenades).
 ->  Error de lectura a 'posicions_jugadors21.json': Expecting ',' delimiter: line 1 column 1048577 (char 1048576)
 ->  Error de lectura a 'posicions_jugadors22.json': Expecting ',' delimiter: line 1 column 2097153 (char 2097152)
 -> Fusionat amb èxit: 'posicions_jugadors23.json' (+47

In [11]:
def identificar_equip_parcial(clau_jugador):

    nom = clau_jugador.lower().strip()

    for jugador in llista_lleida:
        if jugador.lower().strip() in nom:
            return "Hiopos Lleida"

    for jugador in llista_joventut:
        if jugador.lower().strip() in nom:
            return "Joventut de Badalona"

    return "Desconegut"

In [ ]:

# ==============================================================================
# DETECCIÓ AUTOMÀTICA D'ENTORN (Kaggle / Colab / Local)
# ==============================================================================
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    # Entorn Kaggle
    DIRECTORI_BASE = '/kaggle/working/'
elif 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    # Entorn Google Colab
    DIRECTORI_BASE = '/content/'
else:
    # Entorn Local (PC)
    DIRECTORI_BASE = os.getcwd()

# Definim les rutes dinàmicament utilitzant el directori base detectat
RUTA_SORTIDA = os.path.join(DIRECTORI_BASE, 'heatmaps_output')
NOM_ZIP_DESCARREGA = os.path.join(DIRECTORI_BASE, 'heatmaps_complets')

# Creem la carpeta de sortida de manera segura
os.makedirs(RUTA_SORTIDA, exist_ok=True)

print(f"-> Entorn detectat correctament.")
print(f"-> Els heatmaps es desaran a: {RUTA_SORTIDA}")
print(f"-> L'arxiu ZIP es crearà a: {NOM_ZIP_DESCARREGA}.zip\n")

# ==============================================================================
# FUNCIÓ GENERADORA DE HEATMAPS
# ==============================================================================
def generar_heatmap_dades(dades_dict, equip_filtre=None, jugador_filtre=None):
    x_punts = []
    y_punts = []
    coordenades_filtrades = []

    for clau_jugador, coordenades in dades_dict.items():
        # Filtre per jugador mitjançant coincidència parcial
        if jugador_filtre and (jugador_filtre.lower() not in clau_jugador.lower()):
            continue

        equip_detectat = identificar_equip_parcial(clau_jugador)
        if equip_filtre and (equip_detectat != equip_filtre):
            continue

        coordenades_filtrades.extend(coordenades)

    # CORREGIT: Definim correctament el total de punts abans d'usar-lo
    total_punts_acumulats = len(coordenades_filtrades)

    # Limitem a un màxim de 10.000 punts de mostra per a un càlcul gaussià ràpid
    MAX_PUNTS = 10000
    if total_punts_acumulats > MAX_PUNTS:
        pas = total_punts_acumulats // MAX_PUNTS
        coordenades_filtrades = coordenades_filtrades[::pas]

    for punt in coordenades_filtrades:
        if len(punt) >= 2:
            px, py = punt[0], punt[1]
            x_punts.append(px)
            y_punts.append(py)

    # Es valida si hi ha prou dades per fer la interpolació gaussiana de Seaborn
    if len(x_punts) < 15:
        return False

    fig, ax = plt.subplots(figsize=(14, 8))

    # Comprovació del frame de fons (assegura't que FRAME_FONS existeix globalment)
    if 'FRAME_FONS' in globals() and FRAME_FONS is not None:
        ax.imshow(FRAME_FONS)
        amplada = FRAME_FONS.shape[1]
        alcada = FRAME_FONS.shape[0]
        ax.set_xlim(0, amplada)
        ax.set_ylim(alcada, 0)
    else:
        # Fons neutre fosc de reserva si no hi ha vídeo carregat
        ax.set_facecolor('#111111')
        ax.set_xlim(0, 1920)
        ax.set_ylim(1080, 0)

    # Dibuixem el mapa de calor
    sns.kdeplot(
        x=np.array(x_punts), y=np.array(y_punts),
        fill=True,
        thresh=0.08,
        alpha=0.6,
        cmap='rocket',
        bw_adjust=0.5,
        n_levels=15,
        ax=ax
    )

    titol = "Distribució de Densitat Posicional Real (Perspectiva de TV)"
    nom_arxiu = "heatmap_global.png"
    if equip_filtre:
        titol += f"\nEquip: {equip_filtre}"
        nom_arxiu = f"heatmap_{equip_filtre.lower().replace(' ', '_')}.png"
    if jugador_filtre:
        titol += f" (Filtre Jugador: '{jugador_filtre}')"
        nom_arxiu = f"heatmap_{jugador_filtre.lower()}.png"

    plt.title(titol, fontsize=13, fontweight='bold', pad=12)
    ax.axis('off')

    ruta_desar = os.path.join(RUTA_SORTIDA, nom_arxiu)
    plt.savefig(ruta_desar, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    return True


print("\n==================================================")
print("Generant mapa global del partit sobre el frame real...")
print("==================================================")
generar_heatmap_dades(dades_posicions)

print("\n==================================================")
print("Generant mapa de l'equip: Hiopos Lleida...")
print("==================================================")
generar_heatmap_dades(dades_posicions, equip_filtre="Hiopos Lleida")

print("\n==================================================")
print("Generant mapa de l'equip: Joventut de Badalona...")
print("==================================================")
generar_heatmap_dades(dades_posicions, equip_filtre="Joventut de Badalona")

print("\n==================================================")
print("INICIANT GENERACIÓ INTEGRAL PER JUGADORS...")
print("==================================================")
jugadors_totals = sorted(list(dades_posicions.keys()))
generats_amb_exit = 0

for j in jugadors_totals:
    print(f"\nGenerant mapa del jugador: {j}...")

    fet = generar_heatmap_dades(dades_posicions, jugador_filtre=j)
    if fet:
        print(f" -> Mapa de {j} generat, visualitzat i desat correctament.")
        generats_amb_exit += 1
    else:
        print(f" -> Punts de tracking insuficients (<15) per a {j}. S'ha omès la generació.")

print(f"\nS'han generat {generats_amb_exit} mapes de calor individuals dins la carpeta '{RUTA_SORTIDA}'.")
print("\n--- COMPRIMINT TOTS ELS HEATMAPS GENERATS ---")

if os.path.exists(RUTA_SORTIDA) and len(os.listdir(RUTA_SORTIDA)) > 0:
    shutil.make_archive(NOM_ZIP_DESCARREGA, 'zip', RUTA_SORTIDA)
    print("=" * 70)
    print(f"[ÈXIT MESTRE DE DESCÀRREGA]")
    print(f"S'ha creat correctament l'arxiu: '{NOM_ZIP_DESCARREGA}.zip'")
    print("Ja el pots descarregar directament des del teu panell de fitxers!")
    print("=" * 70)
else:
    print("Error. La carpeta de sortida és buida.")

-> Entorn detectat correctament.
-> Els heatmaps es desaran a: /content/heatmaps_output
-> L'arxiu ZIP es crearà a: /content/heatmaps_complets.zip


Generant mapa global del partit sobre el frame real...
